In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

In [ ]:
import h5py

def load_h5py_file(file_path):
    data = {
        'neural_features': [],
        'n_time_steps': [],
        'seq_class_ids': [],
        'seq_len': [],
        'transcriptions': [],
        'sentence_label': [],
        'session': [],
        'block_num': [],
        'trial_num': [],
    }
    # Open the hdf5 file for that day
    with h5py.File(file_path, 'r') as f:

        keys = list(f.keys())

        # For each trial in the selected trials in that day
        for key in keys:
            g = f[key]

            neural_features = g['input_features'][:]
            n_time_steps = g.attrs['n_time_steps']
            seq_class_ids = g['seq_class_ids'][:] if 'seq_class_ids' in g else None
            seq_len = g.attrs['seq_len'] if 'seq_len' in g.attrs else None
            transcription = g['transcription'][:] if 'transcription' in g else None
            sentence_label = g.attrs['sentence_label'][:] if 'sentence_label' in g.attrs else None
            session = g.attrs['session']
            block_num = g.attrs['block_num']
            trial_num = g.attrs['trial_num']

            data['neural_features'].append(neural_features)
            data['n_time_steps'].append(n_time_steps)
            data['seq_class_ids'].append(seq_class_ids)
            data['seq_len'].append(seq_len)
            data['transcriptions'].append(transcription)
            data['sentence_label'].append(sentence_label)
            data['session'].append(session)
            data['block_num'].append(block_num)
            data['trial_num'].append(trial_num)
    return data

### Load data

In [ ]:
import os

data = []
data_path = 'data/t15_copyTask_neuralData/hdf5_data_final'

sessions = os.listdir(data_path)
sessions.sort()
for session in tqdm(sessions):
    # Skip session when no test, train, val
    if len(os.listdir(os.path.join(data_path, session))) < 3:
        continue
    else:
        data.append([])

    session_files = os.listdir(os.path.join(data_path, session))
    session_files.sort()
    for file in session_files:
        file_path = os.path.join(data_path, session, file)
        data[-1].append(load_h5py_file(file_path))
    # print(f"Loaded session {session}")

data[session][test/train/val]

In [ ]:
# first session and test
# data[0][1]

In [ ]:
plt.plot(data[0][1]['neural_features'][1][:5])
plt.show()

## Baseline Approach: nejm-brain-to-text

The baseline model provided in the `nejm-brain-to-text` repository is a **Recurrent Neural Network (RNN)** designed to decode speech from neural activity. Here is a summary of the approach:

### Model Architecture
*   **Input Layer**: Day-specific linear input layers ($512 \times 512$) with **Softsign** activation. This handles non-stationarities in neural recordings across different days.
*   **Recurrent Layers**: A 5-layer **Gated Recurrent Unit (GRU)** network with 768 hidden units per layer.
*   **Output Layer**: A linear projection to phoneme probabilities.

### Training Strategy
*   **Objective**: The model is trained to predict phoneme sequences from neural features using **CTC (Connectionist Temporal Classification) Loss**.
*   **Optimizer**: **AdamW** optimizer.
*   **Data Augmentation**: To improve robustness, the training data is augmented with:
    *   **White Noise**: Added to the neural features.
    *   **Temporal Jitter**: Randomly shifting the neural data in time.
*   **Input Features**: 512 neural features (threshold crossings and spike band power from 256 electrodes), binned at 20 ms resolution.

### Decoding & Inference
*   The RNN outputs **phoneme logits**.
*   These logits are passed to an **n-gram Language Model** (e.g., 3-gram or 5-gram) to decode the most likely sequence of words.
*   The decoding process involves beam search and optional rescoring.

## Model Implementation

Below is the PyTorch implementation of the `GRUDecoder` used in the baseline. It includes:
- **Day-specific input layers** to handle session variability.
- A **multi-layer GRU** backbone.
- An optional **"patching" mechanism** (stacking time steps) to increase the effective context window.

In [ ]:
import torch
from torch import nn

class GRUDecoder(nn.Module):
    '''
    Defines the GRU decoder

    This class combines day-specific input layers, a GRU, and an output classification layer
    '''
    def __init__(self,
                 neural_dim,
                 n_units,
                 n_days,
                 n_classes,
                 rnn_dropout = 0.0,
                 input_dropout = 0.0,
                 n_layers = 5, 
                 patch_size = 0,
                 patch_stride = 0,
                 ):
        '''
        neural_dim  (int)      - number of channels in a single timestep (e.g. 512)
        n_units     (int)      - number of hidden units in each recurrent layer - equal to the size of the hidden state
        n_days      (int)      - number of days in the dataset
        n_classes   (int)      - number of classes 
        rnn_dropout    (float) - percentage of units to droupout during training
        input_dropout (float)  - percentage of input units to dropout during training
        n_layers    (int)      - number of recurrent layers 
        patch_size  (int)      - the number of timesteps to concat on initial input layer - a value of 0 will disable this "input concat" step 
        patch_stride(int)      - the number of timesteps to stride over when concatenating initial input 
        '''
        super(GRUDecoder, self).__init__()
        
        self.neural_dim = neural_dim
        self.n_units = n_units
        self.n_classes = n_classes
        self.n_layers = n_layers 
        self.n_days = n_days

        self.rnn_dropout = rnn_dropout
        self.input_dropout = input_dropout
        
        self.patch_size = patch_size
        self.patch_stride = patch_stride

        # Parameters for the day-specific input layers
        self.day_layer_activation = nn.Softsign() # basically a shallower tanh 

        # Set weights for day layers to be identity matrices so the model can learn its own day-specific transformations
        self.day_weights = nn.ParameterList(
            [nn.Parameter(torch.eye(self.neural_dim)) for _ in range(self.n_days)]
        )
        self.day_biases = nn.ParameterList(
            [nn.Parameter(torch.zeros(1, self.neural_dim)) for _ in range(self.n_days)]
        )

        self.day_layer_dropout = nn.Dropout(input_dropout)
        
        self.input_size = self.neural_dim

        # If we are using "strided inputs", then the input size of the first recurrent layer will actually be in_size * patch_size
        if self.patch_size > 0:
            self.input_size *= self.patch_size

        self.gru = nn.GRU(
            input_size = self.input_size,
            hidden_size = self.n_units,
            num_layers = self.n_layers,
            dropout = self.rnn_dropout, 
            batch_first = True, # The first dim of our input is the batch dim
            bidirectional = False,
        )

        # Set recurrent units to have orthogonal param init and input layers to have xavier init
        for name, param in self.gru.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)

        # Prediciton head. Weight init to xavier
        self.out = nn.Linear(self.n_units, self.n_classes)
        nn.init.xavier_uniform_(self.out.weight)

        # Learnable initial hidden states
        self.h0 = nn.Parameter(nn.init.xavier_uniform_(torch.zeros(1, 1, self.n_units)))

    def forward(self, x, day_idx, states = None, return_state = False):
        '''
        x        (tensor)  - batch of examples (trials) of shape: (batch_size, time_series_length, neural_dim)
        day_idx  (tensor)  - tensor which is a list of day indexs corresponding to the day of each example in the batch x. 
        '''

        # Apply day-specific layer to (hopefully) project neural data from the different days to the same latent space
        day_weights = torch.stack([self.day_weights[i] for i in day_idx], dim=0)
        day_biases = torch.cat([self.day_biases[i] for i in day_idx], dim=0).unsqueeze(1)

        x = torch.einsum("btd,bdk->btk", x, day_weights) + day_biases
        x = self.day_layer_activation(x)

        # Apply dropout to the ouput of the day specific layer
        if self.input_dropout > 0:
            x = self.day_layer_dropout(x)

        # (Optionally) Perform input concat operation
        if self.patch_size > 0: 
  
            x = x.unsqueeze(1)                      # [batches, 1, timesteps, feature_dim]
            x = x.permute(0, 3, 1, 2)               # [batches, feature_dim, 1, timesteps]
            
            # Extract patches using unfold (sliding window)
            x_unfold = x.unfold(3, self.patch_size, self.patch_stride)  # [batches, feature_dim, 1, num_patches, patch_size]
            
            # Remove dummy height dimension and rearrange dimensions
            x_unfold = x_unfold.squeeze(2)           # [batches, feature_dum, num_patches, patch_size]
            x_unfold = x_unfold.permute(0, 2, 3, 1)  # [batches, num_patches, patch_size, feature_dim]

            # Flatten last two dimensions (patch_size and features)
            x = x_unfold.reshape(x.size(0), x_unfold.size(1), -1) 
        
        # Determine initial hidden states
        if states is None:
            states = self.h0.expand(self.n_layers, x.shape[0], self.n_units).contiguous()

        # Pass input through RNN 
        output, hidden_states = self.gru(x, states)

        # Compute logits
        logits = self.out(output)
        
        if return_state:
            return logits, hidden_states
        
        return logits

In [ ]:
# Example instantiation based on baseline hyperparameters
model = GRUDecoder(
    neural_dim=512,
    n_units=768,
    n_days=len(data), # Assuming 'sessions' list is available from previous cells
    n_classes=41,         # 41 classes (phonemes + blank/silence)
    rnn_dropout=0.4,
    input_dropout=0.2,
    n_layers=5,
    patch_size=14,
    patch_stride=4
)

model_cpu = GRUDecoder(
    neural_dim=512,
    n_units=256,
    n_days=len(data), # Assuming 'sessions' list is available from previous cells
    n_classes=41,         # 41 classes (phonemes + blank/silence)
    rnn_dropout=0.4,
    input_dropout=0.2,
    n_layers=2,
    patch_size=14,
    patch_stride=4
)

print(model)
# summary(model, input_size=(2, 100, 512), batch_size=2, device='cpu')

In [ ]:
class CNNGRUDecoder(nn.Module):
    '''
    A Decoder that adds a 1D-CNN feature extractor before the GRU.
    Structure: DayLayer -> Conv1D -> BatchNorm -> ReLU -> GRU -> Linear
    '''
    def __init__(self, neural_dim, n_units, n_days, n_classes, n_layers=5, dropout=0.4, cnn_kernel=3, cnn_stride=1):
        super(CNNGRUDecoder, self).__init__()
        self.n_days = n_days
        self.neural_dim = neural_dim
        self.cnn_stride = cnn_stride # Store for length calc in training loop
        
        # 1. Day-specific Input Layers
        self.day_weights = nn.ParameterList([nn.Parameter(torch.eye(neural_dim)) for _ in range(n_days)])
        self.day_biases = nn.ParameterList([nn.Parameter(torch.zeros(1, neural_dim)) for _ in range(n_days)])
        self.day_act = nn.Softsign()
        self.dropout = nn.Dropout(dropout)
        
        # 2. 1D CNN Block
        # Preserves dimension if stride=1, Downsamples if stride>1
        self.conv1 = nn.Conv1d(neural_dim, neural_dim, kernel_size=cnn_kernel, stride=cnn_stride, padding=cnn_kernel//2)
        self.bn1 = nn.BatchNorm1d(neural_dim)
        self.relu = nn.ReLU()
        
        # 3. GRU Backbone
        self.gru = nn.GRU(neural_dim, n_units, num_layers=n_layers, batch_first=True, dropout=dropout)
        
        # 4. Output Head
        self.fc = nn.Linear(n_units, n_classes)
        
    def forward(self, x, day_idx):
        # x: [Batch, Time, Feat]
        
        # Apply Day-specific projection
        w = torch.stack([self.day_weights[i] for i in day_idx])
        b = torch.cat([self.day_biases[i] for i in day_idx]).unsqueeze(1)
        x = torch.einsum("btd,bdk->btk", x, w) + b
        x = self.day_act(x)
        x = self.dropout(x)
        
        # Apply CNN (Requires [Batch, Feat, Time])
        x = x.permute(0, 2, 1) 
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = x.permute(0, 2, 1) # Back to [Batch, Time, Feat]
        
        # Apply GRU
        x, _ = self.gru(x)
        
        # Output
        x = self.fc(x)
        return x

# Instantiate the new CNN-GRU Model
# We use stride=4 to match the baseline's "patch_stride=4" downsampling.
# This reduces the sequence length by 4x, making it faster and easier to learn.
# We use kernel=7 to capture more local context (similar to patch_size).
model = CNNGRUDecoder(
    neural_dim=512,
    n_units=768,
    n_days=len(data),
    n_classes=41,
    n_layers=5,
    dropout=0.4,
    cnn_kernel=7, 
    cnn_stride=4 
)

model_cpu = CNNGRUDecoder(
    neural_dim=512,
    n_units=256,
    n_days=len(data),
    n_classes=41,
    n_layers=3,
    dropout=0.4,
    cnn_kernel=7,
    cnn_stride=4 
)

## Running Inference with Loaded Data

To use the model with the loaded data, you need to:
1.  **Select a trial**: Extract the `neural_features` for a specific trial.
2.  **Format the input**: Convert the numpy array to a PyTorch tensor and add a batch dimension (shape: `[1, time_steps, 512]`).
3.  **Prepare `day_idx`**: Create a tensor containing the session index for the trial (shape: `[1]`).
4.  **Pass to model**: Call the model with the input and day index.

Note: Since the model is initialized with random weights, the output logits will be random. To get meaningful predictions, you would need to load the pretrained weights.

In [ ]:
# 1. Select a trial (e.g., first session, first file, first trial)
session_idx = 0
file_idx = 0 # Usually 0=train, 1=test, 2=val depending on sorting, check data[session_idx][file_idx].keys() if unsure
trial_idx = 0

# Get neural features: shape (time_steps, 512)
features_numpy = data[session_idx][file_idx]['neural_features'][trial_idx]

# 2. Format input
# Convert to tensor and add batch dimension -> (1, time_steps, 512)
x = torch.tensor(features_numpy, dtype=torch.float32).unsqueeze(0)

# 3. Prepare day_idx
# The model needs to know which day/session this data belongs to for the day-specific layer
day_idx = torch.tensor([session_idx], dtype=torch.long)

# 4. Run the model
model.eval() # Set to evaluation mode
with torch.no_grad():
    logits = model(x, day_idx)

print(f"Input shape: {x.shape}")
print(f"Output logits shape: {logits.shape}") # Should be (1, output_time_steps, n_classes)
print("Inference successful!")

## Training Loop Example

Here is a cell that demonstrates how to train the model on a few examples from the loaded data.
It defines the **CTC Loss**, the **Optimizer**, and runs a training step.

In [ ]:
import torch.optim as optim
import random
import numpy as np

# --- Hyperparameters ---
learning_rate = 5e-4   # <--- Increased LR for faster convergence
n_epochs = 10          # <--- Increased Epochs
weight_decay = 1e-2
noise_std = 0.2        
max_sessions = 3       # <--- Train on ONLY 1 session for speed
max_val_trials = 10    # <--- Validate on ONLY 10 trials

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cpu':
    print("Warning: Training on CPU will be slow model changed to a cpu friendly one.")
    model = model_cpu

# --- Setup ---
model.to(device)
ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=False)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

train_losses = []
val_losses = []

os.makedirs("CNNGRU_model_weights", exist_ok=True)

print(f"Starting training on {min(len(data), max_sessions)} sessions...")

for epoch in range(n_epochs):
    # --- TRAINING PHASE ---
    model.train()
    total_train_loss = 0
    train_batches = 0
    
    for session_idx, session_data in enumerate(data):
        if session_idx >= max_sessions: break

        if len(session_data) < 3: continue
        train_data = session_data[1] 
        num_trials = len(train_data['neural_features'])
        
        trial_indices = list(range(num_trials))
        random.shuffle(trial_indices)
        
        for i in trial_indices:
            features = train_data['neural_features'][i]
            targets = train_data['seq_class_ids'][i]
            target_len = train_data['seq_len'][i]
            n_time_steps = features.shape[0]
            
            # Validation checks
            if targets is None or target_len > len(targets): continue
            
            # Trim targets and check for blank token
            targets = targets[:target_len]
            if 0 in targets: continue 

            # --- Data Augmentation & Normalization ---
            features_tensor = torch.tensor(features, dtype=torch.float32)
            
            # Z-Score Normalization (Crucial for convergence)
            # Normalize per channel over time
            mean = features_tensor.mean(dim=0, keepdim=True)
            std = features_tensor.std(dim=0, keepdim=True)
            features_tensor = (features_tensor - mean) / (std + 1e-8)

            # Add Gaussian Noise
            features_tensor = features_tensor + torch.randn_like(features_tensor) * noise_std
            
            # Prepare Inputs and move to Device
            x = features_tensor.unsqueeze(0).to(device)
            day_idx = torch.tensor([session_idx], dtype=torch.long).to(device)
            y = torch.tensor(targets, dtype=torch.long).to(device)
            
            # Calculate input lengths (Dynamic based on model type)
            if hasattr(model, 'patch_size') and model.patch_size > 0:
                input_len_val = int((n_time_steps - model.patch_size) / model.patch_stride + 1)
            elif hasattr(model, 'cnn_stride'):
                # More precise calculation for Conv1d output length
                padding = model.conv1.padding[0]
                kernel = model.conv1.kernel_size[0]
                stride = model.conv1.stride[0]
                input_len_val = int((n_time_steps + 2*padding - (kernel-1) - 1)/stride + 1)
            else:
                input_len_val = n_time_steps
            
            if input_len_val < target_len: continue

            input_lengths = torch.tensor([input_len_val], dtype=torch.long)
            target_lengths = torch.tensor([target_len], dtype=torch.long)

            # Optimization
            optimizer.zero_grad()
            logits = model(x, day_idx)
            log_probs = logits.log_softmax(2).permute(1, 0, 2)
            
            try:
                loss = ctc_loss(log_probs, y, input_lengths, target_lengths)
                if not torch.isnan(loss) and not torch.isinf(loss):
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
                    optimizer.step()
                    
                    total_train_loss += loss.item()
                    train_batches += 1
                    
                    # Progress print
                    if train_batches % 10 == 0:
                        print(f"  Epoch {epoch+1} | Batch {train_batches}/{num_trials} | Loss: {loss.item():.4f}", end='\r')
            except Exception:
                continue
    
    print() # Newline after progress bar
    avg_train_loss = total_train_loss / train_batches if train_batches > 0 else 0
    train_losses.append(avg_train_loss)
    
    # --- VALIDATION PHASE ---
    model.eval()
    total_val_loss = 0
    val_batches = 0
    
    with torch.no_grad():
        for session_idx, session_data in enumerate(data):
            if session_idx >= max_sessions: break 

            if len(session_data) < 3: continue
            val_data = session_data[2] 
            
            num_trials = len(val_data['neural_features'])
            val_indices = list(range(num_trials))
            if num_trials > max_val_trials:
                val_indices = random.sample(val_indices, max_val_trials)
            
            for i in val_indices:
                features = val_data['neural_features'][i]
                targets = val_data['seq_class_ids'][i]
                target_len = val_data['seq_len'][i]
                n_time_steps = features.shape[0]
                
                if targets is None or target_len > len(targets): continue
                targets = targets[:target_len]
                if 0 in targets: continue

                # Normalize Validation Data
                features_tensor = torch.tensor(features, dtype=torch.float32)
                mean = features_tensor.mean(dim=0, keepdim=True)
                std = features_tensor.std(dim=0, keepdim=True)
                features_tensor = (features_tensor - mean) / (std + 1e-8)

                x = features_tensor.unsqueeze(0).to(device)
                day_idx = torch.tensor([session_idx], dtype=torch.long).to(device)
                y = torch.tensor(targets, dtype=torch.long).to(device)
                
                # Calculate input lengths (Dynamic)
                if hasattr(model, 'patch_size') and model.patch_size > 0:
                    input_len_val = int((n_time_steps - model.patch_size) / model.patch_stride + 1)
                elif hasattr(model, 'cnn_stride'):
                    padding = model.conv1.padding[0]
                    kernel = model.conv1.kernel_size[0]
                    stride = model.conv1.stride[0]
                    input_len_val = int((n_time_steps + 2*padding - (kernel-1) - 1)/stride + 1)
                else:
                    input_len_val = n_time_steps

                if input_len_val < target_len: continue

                input_lengths = torch.tensor([input_len_val], dtype=torch.long)
                target_lengths = torch.tensor([target_len], dtype=torch.long)
                
                logits = model(x, day_idx)
                log_probs = logits.log_softmax(2).permute(1, 0, 2)
                
                try:
                    loss = ctc_loss(log_probs, y, input_lengths, target_lengths)
                    total_val_loss += loss.item()
                    val_batches += 1
                except: continue

    avg_val_loss = total_val_loss / val_batches if val_batches > 0 else 0
    val_losses.append(avg_val_loss)
    
    scheduler.step(avg_val_loss)

    torch.save(model.state_dict(), f"CNNGRU_model_weights/model_epoch_{epoch+1}.pth")
    
    print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

print("Training complete.")

In [ ]:
torch.save(model, 'cnn_gru_decoder_model.pth')

In [ ]:
# Plot Training vs Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('CTC Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
LOGIT_TO_PHONEME = [
'BLANK',    # "BLANK" = CTC blank symbol
'AA', 'AE', 'AH', 'AO', 'AW',
'AY', 'B', 'CH', 'D', 'DH',
'EH', 'ER', 'EY', 'F', 'G',
'HH', 'IH', 'IY', 'JH', 'K',
'L', 'M', 'N', 'NG', 'OW',
'OY', 'P', 'R', 'S', 'SH',
'T', 'TH', 'UH', 'UW', 'V',
'W', 'Y', 'Z', 'ZH',
' | ',    # "|" = silence token
]

def indexes_to_phonemes(indexes):
    phonemes = [LOGIT_TO_PHONEME[idx] for idx in indexes if idx in range(len(LOGIT_TO_PHONEME))]
    return phonemes

In [ ]:
indexes_to_phonemes(data[0][1]['seq_class_ids'][0][:13]), data[0][1]['sentence_label'][0]

## Phoneme-to-Text Translation Model

To translate the sequence of phonemes (which may contain errors) into coherent English sentences, a **Sequence-to-Sequence (Seq2Seq) Transformer** is a powerful choice.

### Why a Transformer?
1.  **Error Correction**: By training on pairs of `(Noisy Phonemes, Correct Text)`, the model learns to correct common decoding errors (e.g., fixing "hello" -> "hello" if the phoneme for 'h' was missed).
2.  **Context Awareness**: It uses the entire sequence context to resolve ambiguities (e.g., "two" vs "too" vs "to" based on surrounding phonemes).
3.  **Variable Lengths**: It handles the fact that the number of phonemes does not match the number of characters/words 1-to-1.

### Implementation
Below is a PyTorch implementation of a Transformer designed for this task. 
*   **Input**: Sequence of phoneme indices (from your `GRUDecoder` output).
*   **Output**: Sequence of text tokens (characters or subwords).

**Note on Training**: To make the model robust to errors ("know the errors"), you should apply **augmentation** to the input phonemes during training (randomly substituting, deleting, or inserting phonemes) if you are training on ground truth phonemes. Ideally, you would fine-tune on the actual outputs of your `GRUDecoder`.

In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class PhonemeToTextTransformer(nn.Module):
    def __init__(self, num_phonemes, num_text_tokens, d_model=256, nhead=4, num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=1024, dropout=0.1):
        super(PhonemeToTextTransformer, self).__init__()
        self.d_model = d_model
        
        # Embeddings
        # num_phonemes: Size of your phoneme vocabulary (e.g., 41)
        # num_text_tokens: Size of your text vocabulary (e.g., 30 for char-level: a-z, space, punctuation)
        self.phoneme_embedding = nn.Embedding(num_phonemes, d_model)
        self.text_embedding = nn.Embedding(num_text_tokens, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # Transformer
        # batch_first=True is easier to work with
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead, 
                                          num_encoder_layers=num_encoder_layers, 
                                          num_decoder_layers=num_decoder_layers, 
                                          dim_feedforward=dim_feedforward, 
                                          dropout=dropout, batch_first=True)
        
        # Output Head
        self.fc_out = nn.Linear(d_model, num_text_tokens)
        
    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        '''
        src: [batch_size, src_len] (Phoneme indices)
        tgt: [batch_size, tgt_len] (Text token indices)
        '''
        
        # Generate mask to prevent decoder from looking ahead
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        
        # Embed + Positional Encoding
        src_emb = self.pos_encoder(self.phoneme_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.text_embedding(tgt) * math.sqrt(self.d_model))
        
        # Transformer Pass
        outs = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask, 
                                src_key_padding_mask=src_key_padding_mask, 
                                tgt_key_padding_mask=tgt_key_padding_mask, 
                                memory_key_padding_mask=memory_key_padding_mask)
        
        # Project to vocabulary size
        return self.fc_out(outs)

# Example Instantiation
# Assuming 41 phonemes and a simple character-level text vocabulary of ~500 characters
p2t_model = PhonemeToTextTransformer(
    num_phonemes=41, 
    num_text_tokens=500, 
    d_model=128, 
    nhead=4, 
    num_encoder_layers=2, 
    num_decoder_layers=2
)

print(p2t_model)


## Training the Transformer

To train the `PhonemeToTextTransformer`, we need to:
1.  **Tokenize the Text**: Convert English sentences into sequences of integers (indices).
2.  **Prepare the Data**: Create a PyTorch `Dataset` and `DataLoader` to batch the pairs of `(Phoneme Sequence, Text Sequence)`.
3.  **Run the Training Loop**: Optimize the model to predict the next text token given the phonemes and previous text tokens.

### 1. Text Tokenizer
We'll use a simple character-level tokenizer.

In [ ]:
class CharTokenizer:
    def __init__(self):
        # 0: Pad, 1: Start of Sentence (SOS), 2: End of Sentence (EOS)
        self.char2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2}
        self.idx2char = {0: '<pad>', 1: '<sos>', 2: '<eos>'}
        self.vocab_size = 3

    def fit(self, sentences):
        unique_chars = set("".join(sentences))
        for char in sorted(unique_chars):
            if char not in self.char2idx:
                self.char2idx[char] = self.vocab_size
                self.idx2char[self.vocab_size] = char
                self.vocab_size += 1
    
    def encode(self, sentence):
        # Add SOS at start and EOS at end
        return [self.char2idx['<sos>']] + [self.char2idx[c] for c in sentence] + [self.char2idx['<eos>']]

    def decode(self, indices):
        # Convert back to string, ignoring special tokens
        return "".join([self.idx2char[idx] for idx in indices if idx not in [0, 1, 2]])

# Collect all sentences to build vocabulary
all_sentences = []
for session_data in data:
    if len(session_data) > 1: # Ensure train set exists
        # Handle bytes vs str
        sentences = [s.decode('utf-8') if isinstance(s, bytes) else s for s in session_data[1]['sentence_label']]
        all_sentences.extend(sentences)

# Initialize and fit tokenizer
tokenizer = CharTokenizer()
tokenizer.fit(all_sentences)
print(f"Vocabulary Size: {tokenizer.vocab_size}")
print(f"Example encoding: 'hello' -> {tokenizer.encode('hello')}")

### 2. Dataset and DataLoader
We create a custom Dataset to handle the pairing of phonemes and text.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class PhonemeTextDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.samples = []
        self.tokenizer = tokenizer
        
        # Flatten the data structure
        for session in data:
            if len(session) < 2: continue
            train_data = session[1] # 1 is train
            
            for i in range(len(train_data['seq_class_ids'])):
                phonemes = train_data['seq_class_ids'][i]
                sentence = train_data['sentence_label'][i]
                
                if phonemes is None or sentence is None: continue
                
                # Clean sentence
                if isinstance(sentence, bytes): 
                    sentence = sentence.decode('utf-8')
                
                self.samples.append((phonemes, sentence))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        phonemes, sentence = self.samples[idx]
        
        # Convert to tensors
        # Phonemes: already indices. 
        # Note: If your phonemes have a 'blank' token 0, it might conflict with padding 0. 
        # Ideally, ensure padding is a unique index. Here we assume 0 is safe or handled.
        src = torch.tensor(phonemes, dtype=torch.long)
        
        # Text: Encode to indices
        tgt = torch.tensor(self.tokenizer.encode(sentence), dtype=torch.long)
        
        return src, tgt

def collate_fn(batch):
    # Pad sequences to max length in batch
    src_batch, tgt_batch = zip(*batch)
    
    # Pad with 0 (assuming 0 is <pad> for both, or adjust accordingly)
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=0) 
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=0) 
    
    return src_padded, tgt_padded

# Create Dataset and DataLoader
dataset = PhonemeTextDataset(data, tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

print(f"Dataset size: {len(dataset)}")
first_batch = next(iter(dataloader))
print(f"Batch shapes - Src: {first_batch[0].shape}, Tgt: {first_batch[1].shape}")

### 3. Training Loop
Now we train the model.
*   **Input to Decoder (`tgt_input`)**: The target sequence *excluding* the last token (e.g., `<sos> h e l l o`).
*   **Target for Loss (`tgt_output`)**: The target sequence *excluding* the first token (e.g., `h e l l o <eos>`).
*   **Teacher Forcing**: We feed the correct previous tokens to the decoder during training.

In [ ]:
# --- Hyperparameters ---
EPOCHS = 5
LR = 0.0005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Model Initialization ---
# Re-initialize model with correct vocab size
transformer_model = PhonemeToTextTransformer(
    num_phonemes=41, 
    num_text_tokens=tokenizer.vocab_size,
    d_model=64,
    nhead=2,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dropout=0.1
).to(DEVICE)

# --- Optimizer & Loss ---
optimizer = optim.AdamW(transformer_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore padding in loss calculation

# --- Training Loop ---
print(f"Starting training on {DEVICE}...")

os.makedirs("transformer_model_weights", exist_ok=True)

transformer_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    
    for batch_idx, (src, tgt) in enumerate(dataloader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        
        # Prepare inputs and targets
        # tgt_input: <sos> ... n-1
        tgt_input = tgt[:, :-1]
        
        # tgt_output: 1 ... <eos>
        tgt_output = tgt[:, 1:]
        
        # Create Padding Masks (Optional but recommended for variable lengths)
        # src_padding_mask = (src == 0)
        # tgt_padding_mask = (tgt_input == 0)
        
        optimizer.zero_grad()
        
        # Forward Pass
        # The model handles the look-ahead mask internally
        output = transformer_model(src, tgt_input)
        
        # Reshape for Loss
        # Output: [batch, seq_len, vocab_size] -> [batch * seq_len, vocab_size]
        # Target: [batch, seq_len] -> [batch * seq_len]
        output_flat = output.reshape(-1, output.shape[-1])
        tgt_flat = tgt_output.reshape(-1)
        
        loss = criterion(output_flat, tgt_flat)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 1 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}", end='\r')
            
    avg_loss = total_loss / len(dataloader)

    torch.save(transformer_model.state_dict(), f"transformer_model_weights/model_epoch_{epoch+1}.pth")
    print(f"\nEpoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")

print("Training Finished!")